# Analytical Mantle Flow: Example Workflow

This notebook starts with simple examples and then builds up to a global plate-boundary case from Holt and Royden (2020).

## 0) Setup

- Run this notebook from the repository root environment (`mantle-flow-modeling`).
- Set `RUN_HEAVY = False` to use precomputed outputs only.
- Set `RUN_HEAVY = True` to rerun model scripts.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'flow_computations').exists() else CWD.parent
if not (ROOT / 'flow_computations').exists():
    # fallback: search upward a few levels
    for parent in CWD.parents:
        if (parent / 'flow_computations').exists():
            ROOT = parent
            break
FLOW = ROOT / 'flow_computations'
RUN_HEAVY = False  # flip to True to rerun scripts

if not (ROOT / 'flow_computations').exists():
    raise RuntimeError('Could not locate repository root containing flow_computations/.')

print('Repo root:', ROOT)
print('RUN_HEAVY =', RUN_HEAVY)


In [ ]:
def run(cmd, cwd=FLOW, env=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True, env=env)

def show_image(path):
    path = Path(path)
    if not path.exists():
        print('Missing image:', path)
        return
    img = plt.imread(path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(path.name)
    plt.show()

## 1) Simple Example A: Idealized retreating trench

Model: `LargeSP_RetreatingTrench` (no slab flux).

In [ ]:
if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py',
         'LargeSP_RetreatingTrench', '3.0e20', '0', '500000', '0', '0', 'Subgrd.inp', '4.0e20'])

dp_a = np.loadtxt(FLOW / 'text_files' / 'LargeSP_RetreatingTrench.3e+20noslabflux' / 'DP.txt')
print('DP rows:', dp_a.shape[0])
print('DP mean/median/min/max [MPa]:', np.mean(dp_a[:,4]), np.median(dp_a[:,4]), np.min(dp_a[:,4]), np.max(dp_a[:,4]))
show_image(FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrench.3e+20noslabflux.plotvisc4e+20.png')

## 2) Simple Example B: Idealized slab-gap geometry

Model: `LargeSP_RetreatingTrenchSlabGap` (no slab flux).

In [ ]:
if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py',
         'LargeSP_RetreatingTrenchSlabGap', '3.0e20', '0', '500000', '0', '0', 'Subgrd.inp', '4.0e20'])

dp_b = np.loadtxt(FLOW / 'text_files' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux' / 'DP.txt')
print('DP rows:', dp_b.shape[0])
print('DP mean/median/min/max [MPa]:', np.mean(dp_b[:,4]), np.median(dp_b[:,4]), np.min(dp_b[:,4]), np.max(dp_b[:,4]))
show_image(FLOW / 'plots' / 'pressure_fields' / 'LargeSP_RetreatingTrenchSlabGap.3e+20noslabflux.plotvisc4e+20.png')

## 3) Build up to global case

Case: `Slab2.0Final_NoJapTail_nnr_FS` with convergence-scaled slab flux.

Steps:
1. run global pressure model
2. compute subducting plate ages
3. compare modeled vs observed dips

In [ ]:
global_model = 'Slab2.0Final_NoJapTail_nnr_FS'
global_text = FLOW / 'text_files' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux'
global_plot = FLOW / 'plots' / 'pressure_fields' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.plotvisc4e+20.png'
global_dip_plot = FLOW / 'plots' / 'dip_comparisons' / f'{global_model}.3e+20_VcSlabFlux_width500000.0_alpha0.0.NoTailFlux.fact1.327.png'

env = os.environ.copy()
env['DIPS_OBS_TXT'] = str(ROOT / 'dip_observations' / 'dip_catalogues' / 'Slab2_const-depth' / 'AllDips.txt')

if RUN_HEAVY:
    run([sys.executable, 'global_pressure_withPressurePlot.py', global_model, '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '4.0e20'], env=env)
    run([sys.executable, 'get_SPages.py', global_model], env=env)
    run([sys.executable, 'plot_DipComparison_varyDPfactor.py', global_model, '3.0e20', '2', '500000', '0', '1', 'Subgrd.inp', '2', '12', '4', '5'], env=env)

dp_global = np.loadtxt(global_text / 'DP.txt')
print('Global DP rows:', dp_global.shape[0])
print('Global DP mean/median/min/max [MPa]:', np.mean(dp_global[:,4]), np.median(dp_global[:,4]), np.min(dp_global[:,4]), np.max(dp_global[:,4]))
show_image(global_plot)
show_image(global_dip_plot)

## 4) Optional: Run via main driver

From a terminal:

```bash
./run_example_workflow.py --mode quick
./run_example_workflow.py --mode full
```